# Chapter 13 companion: designing digital controllers.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Run:  python3 ch13_controller_design.py
Needs numpy, scipy and matplotlib. Prints the numbers of the chapter and saves four plots
in the folder figures/ next to this file:

  ch13_emulation.png     analog lead vs. its Tustin emulation at T = 0.2 s and T = 1 s
  ch13_direct_lead.png   lead D(z) designed on the z-plane (T = 1 s), samples and c(t)
  ch13_deadbeat.png      minimum-time deadbeat (ripple) and ripple-free deadbeat
  ch13_pid.png           digital PID (positional form) on 4/((s+1)(s+4)), T = 0.05 s

The plant of Part A and Part B is G_p(s) = 1/(s(s+1)) driven by a zero-order hold.
Polynomials are numpy-style lists of coefficients in descending powers of s or z.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
# In the browser (JupyterLite), hide a harmless Matplotlib warning from its in-browser backend.
import sys, warnings
if "pyodide" in sys.modules:
    warnings.filterwarnings("ignore", message=r"The \w+ parameter as float was deprecated")

In [ ]:
from IPython.display import display
import os

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm
from scipy.optimize import brentq

HERE = os.getcwd()
FIGDIR = os.path.join(HERE, "figures")
os.makedirs(FIGDIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# 1. Turning G_c(s) into D(z)
# ---------------------------------------------------------------------------
def substitute(num_s, den_s, top, bot):
    """Replace s by top(z)/bot(z) in num_s/den_s and clear the fractions.

    top and bot are polynomials in z of degree <= 1. Forward Euler: s = (z-1)/T, so
    top = [1, -1], bot = [T]. Backward Euler: top = [1, -1], bot = [T, 0].
    Tustin: top = [2/T, -2/T], bot = [1, 1].
    """
    n = max(len(num_s), len(den_s)) - 1

    def convert(p):
        p = np.concatenate([np.zeros(n + 1 - len(p)), p])
        out = np.zeros(1)
        for i, coef in enumerate(p):
            k = n - i  # this coefficient multiplies s^k
            term = np.array([coef])
            for _ in range(k):
                term = np.polymul(term, top)
            for _ in range(n - k):
                term = np.polymul(term, bot)
            out = np.polyadd(out, term)
        return out

    num_z, den_z = convert(np.asarray(num_s, float)), convert(np.asarray(den_s, float))
    return num_z / den_z[0], den_z / den_z[0]

In [ ]:
def forward_euler(num_s, den_s, T):
    return substitute(num_s, den_s, [1.0, -1.0], [T])

In [ ]:
def backward_euler(num_s, den_s, T):
    return substitute(num_s, den_s, [1.0, -1.0], [T, 0.0])

In [ ]:
def tustin(num_s, den_s, T, prewarp=None):
    """Tustin rule; with prewarp = w1 the factor 2/T becomes w1/tan(w1 T/2)."""
    c = 2.0 / T if prewarp is None else prewarp / np.tan(prewarp * T / 2)
    return substitute(num_s, den_s, [c, -c], [1.0, 1.0])

In [ ]:
def matched_pole_zero(num_s, den_s, T):
    """Map poles and zeros by z = e^{sT}, zeros at infinity to z = -1, match the DC gain."""
    zeros, poles = np.roots(num_s), np.roots(den_s)
    zd = list(np.exp(zeros * T)) + [-1.0] * (len(poles) - len(zeros))
    num_z, den_z = np.real(np.poly(zd)), np.real(np.poly(np.exp(poles * T)))
    gain = (np.polyval(num_s, 0) / np.polyval(den_s, 0)) / (np.polyval(num_z, 1) / np.polyval(den_z, 1))
    return gain * num_z, den_z

In [ ]:
def zoh_matrices(A, B, T):
    """Phi = e^{AT} and Gamma = int_0^T e^{A tau} d tau B, from one matrix exponential."""
    n = A.shape[0]
    M = np.zeros((n + 1, n + 1))
    M[:n, :n] = A
    M[:n, n] = B
    P = expm(M * T)
    return P[:n, :n], P[:n, n]

In [ ]:
def zoh_equivalent_first_order(K, a, b, T):
    """ZOH (step-invariant) equivalent of K(s+a)/(s+b) = K + K(a-b)/(s+b).

    State space: x' = -b x + e, u = K(a-b) x + K e.
    """
    Phi, Gam = zoh_matrices(np.array([[-b]]), np.array([1.0]), T)
    phi, gam = Phi[0, 0], Gam[0]
    # D(z) = K + K(a-b) gam/(z - phi)
    return np.array([K, -K * phi + K * (a - b) * gam]), np.array([1.0, -phi])

In [ ]:
# ---------------------------------------------------------------------------
# 2. The plant seen by the computer, and loop simulation
# ---------------------------------------------------------------------------
A_RUN = np.array([[0.0, 1.0], [0.0, -1.0]])  # 1/(s(s+1)): states c and dc/dt

In [ ]:
B_RUN = np.array([0.0, 1.0])
C_RUN = np.array([1.0, 0.0])

In [ ]:
def pulse_tf_running(T):
    """G(z) = (b1 z + b0)/((z-1)(z-e^{-T})) for ZOH + 1/(s(s+1))."""
    p = np.exp(-T)
    return np.array([T - 1 + p, 1 - p - T * p]), np.polymul([1, -1], [1, -p])

In [ ]:
def simulate_loop(Dn, Dd, A, B, C, T, n_samples, fine=20):
    """Unit-step response of the sampled loop r -> e -> D(z) -> ZOH -> plant -> c.

    The controller runs as a difference equation; the plant is propagated exactly between
    samples with matrix exponentials, so c(t) between the samples is exact too.
    Returns: t_k, c(kT), u(kT), t_fine, c(t_fine).
    """
    Dn = np.concatenate([np.zeros(len(Dd) - len(Dn)), Dn]) / Dd[0]
    Dd = np.asarray(Dd, float) / Dd[0]
    order = len(Dd) - 1
    steps = [zoh_matrices(A, B, T * j / fine) for j in range(fine)]
    Phi, Gam = zoh_matrices(A, B, T)
    x = np.zeros(A.shape[0])
    e_hist = np.zeros(order + 1)  # e(k), e(k-1), ...
    u_hist = np.zeros(order + 1)  # u(k), u(k-1), ...
    ck, uk, tf, cf = [], [], [], []
    for k in range(n_samples):
        e_hist = np.roll(e_hist, 1)
        e_hist[0] = 1.0 - C @ x
        u = Dn @ e_hist - Dd[1:] @ u_hist[:-1]
        u_hist = np.roll(u_hist, 1)
        u_hist[0] = u
        ck.append(C @ x)
        uk.append(u)
        for j, (Pj, Gj) in enumerate(steps):
            tf.append((k + j / fine) * T)
            cf.append(C @ (Pj @ x + Gj * u))
        x = Phi @ x + Gam * u
    return np.arange(n_samples) * T, np.array(ck), np.array(uk), np.array(tf), np.array(cf)

In [ ]:
def zeta_of(z):
    """Damping ratio of a z-plane pole (through s = ln(z)/T)."""
    s = np.log(complex(z))
    return -s.real / abs(s)

In [ ]:
# ---------------------------------------------------------------------------
# 3. Part A: one lead compensator, six conversions
# ---------------------------------------------------------------------------
print("=== Part A: emulation of Gc(s) = 4(s+1)/(s+2), T = 0.2 s ===")

In [ ]:
num_c, den_c = [4.0, 4.0], [1.0, 2.0]
T = 0.2
designs = {
    "forward Euler": forward_euler(num_c, den_c, T),
    "backward Euler": backward_euler(num_c, den_c, T),
    "Tustin": tustin(num_c, den_c, T),
    "Tustin, prewarped": tustin(num_c, den_c, T, prewarp=np.sqrt(2)),
    "matched pole-zero": matched_pole_zero(num_c, den_c, T),
    "ZOH equivalent": zoh_equivalent_first_order(4.0, 1.0, 2.0, T),
}
gn, gd = pulse_tf_running(T)

In [ ]:
for name, (Dn, Dd) in designs.items():
    poles = np.roots(np.polyadd(np.polymul(Dd, gd), np.polymul(Dn, gn)))
    pair = poles[np.argmax(poles.imag)]
    _, ck, _, _, _ = simulate_loop(Dn, Dd, A_RUN, B_RUN, C_RUN, T, 200, fine=1)
    print(f"{name:18s} D(z) = {Dn[0]:.4f}(z - {-Dn[1] / Dn[0]:.4f})/(z - {-Dd[1]:.4f})"
          f"   poles {pair.real:.4f} +/- j{pair.imag:.4f}, zeta {zeta_of(pair):.2f},"
          f" overshoot {100 * (ck.max() - 1):.1f}%")

In [ ]:
# the same Tustin recipe at T = 0.2 s and at T = 1 s
fig, ax = plt.subplots(figsize=(7, 3.6))

t = np.linspace(0, 14, 1000)
ax.plot(t, 1 - np.exp(-t) * (np.cos(np.sqrt(3) * t) + np.sin(np.sqrt(3) * t) / np.sqrt(3)),
        "k--", label="analog loop")

for Ts, style in [(0.2, "o"), (1.0, "s")]:
    Dn, Dd = tustin(num_c, den_c, Ts)
    tk, ck, _, tf, cf = simulate_loop(Dn, Dd, A_RUN, B_RUN, C_RUN, Ts, int(14 / Ts), fine=10)
    line, = ax.plot(tf, cf, lw=0.8)
    ax.plot(tk, ck, style, ms=3, color=line.get_color(), label=f"Tustin, T = {Ts} s")
    print(f"Tustin at T = {Ts}: peak of the samples {ck.max():.3f}")

ax.axhline(1, color="gray", lw=0.5)
ax.set_xlabel("t (s)")
ax.set_ylabel("c")
ax.legend()
ax.set_title("Emulation: the same analog lead at two sampling periods")
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch13_emulation.png"), dpi=150)
display(fig)
plt.close(fig)

In [ ]:
# the hold's phase lag at crossover
wc = np.sqrt(2 * np.sqrt(5) - 2)

In [ ]:
pm = 90 - np.degrees(np.arctan(wc / 2))
print(f"crossover {wc:.3f} rad/s, analog PM {pm:.1f} deg")

In [ ]:
for Ts in [0.05, 0.1, 0.2, 0.5, 1.0]:
    lag = np.degrees(wc * Ts / 2)
    print(f"  T = {Ts:4}: hold lag {lag:4.1f} deg, PM estimate {pm - lag:4.1f} deg")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Part B: a lead controller designed on the z-plane (T = 1 s)
# ---------------------------------------------------------------------------
print("\n=== Part B: direct design, T = 1 s ===")

In [ ]:
T = 1.0
gn, gd = pulse_tf_running(T)
c = np.e - 2  # plant zero at -c

In [ ]:
# D(z) = Kc (z - e^-1)/z  ->  1 + DG = 0 is z^2 + (K'-1) z + K' c = 0, K' = Kc e^-1.
# Complex roots lie on |z + c| = sqrt(c(1+c)); intersect with the zeta = 0.5 curve.
theta = brentq(lambda th: np.exp(-th / np.sqrt(3))
               - (-c * np.cos(th) + np.sqrt(c**2 * np.cos(th) ** 2 + c)), 0.1, 2.5)

In [ ]:
r = np.exp(-theta / np.sqrt(3))
Kprime = r**2 / c  # product of the roots is K'c
Kc = Kprime * np.e
print(f"design pole {r * np.cos(theta):.4f} +/- j{r * np.sin(theta):.4f}, "
      f"Kc = {Kc:.3f}, Kv = {Kprime * (np.e - 1):.3f}")
Dn, Dd = np.array([Kc, -Kc * np.exp(-1)]), np.array([1.0, 0.0])
tk, ck, uk, tf, cf = simulate_loop(Dn, Dd, A_RUN, B_RUN, C_RUN, T, 15, fine=20)
print("c(k):", np.round(ck[:11], 4))
print("u(k):", np.round(uk[:8], 4))
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(tf, cf, "r-", lw=1, label="c(t)")
ax.plot(tk, ck, "ro", ms=4, label="c(kT)")
ax.step(tk, uk, where="post", color="tab:blue", lw=1, label="u (held)")
ax.axhline(1, color="gray", lw=0.5)
ax.set_xlabel("t (s)")
ax.legend()
ax.set_title("Lead controller 1.158(z - 0.3679)/z, T = 1 s")
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch13_direct_lead.png"), dpi=150)
display(fig)
plt.close(fig)

In [ ]:
# ---------------------------------------------------------------------------
# 5. Deadbeat: minimum time (ripple) and ripple-free
# ---------------------------------------------------------------------------
def ragazzini(F_num, F_den, gn, gd):
    """D = F/(G(1-F)) for F = F_num/F_den, as polynomials (no cancellation is done)."""
    one_minus_F = np.polysub(F_den, F_num)
    return np.polymul(F_num, gd), np.polymul(gn, one_minus_F)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 5.2), sharex=True)

for name, (Dn, Dd) in {
    "minimum time, F = 1/z": (np.array([np.e, -1.0]), np.array([1.0, c])),
    "ripple-free, F = (z + c)/((e-1) z^2)": (np.e / (np.e - 1) * np.array([1.0, -np.exp(-1)]),
                                             np.array([1.0, (np.e - 2) / (np.e - 1)])),
}.items():
    tk, ck, uk, tf, cf = simulate_loop(Dn, Dd, A_RUN, B_RUN, C_RUN, T, 12, fine=40)
    print(f"{name}: c(k) = {np.round(ck[:4], 4)}, u(k) = {np.round(uk[:4], 4)}, "
          f"peak of c(t) = {cf.max():.3f}")
    line, = axes[0].plot(tf, cf, lw=1, label=name)
    axes[0].plot(tk, ck, "o", ms=3, color=line.get_color())
    axes[1].step(tk, uk, where="post", color=line.get_color(), lw=1)

# the same controllers from Ragazzini's formula, before cancelling common factors
Dn_rf, Dd_rf = ragazzini(np.array([1.0, c]) / (np.e - 1), np.array([1.0, 0.0, 0.0]), gn, gd)

print("ripple-free D(z), uncancelled form: zeros", np.round(np.roots(Dn_rf), 4),
      "poles", np.round(np.roots(Dd_rf), 4))
axes[0].axhline(1, color="gray", lw=0.5)
axes[0].set_ylabel("c")
axes[0].legend(fontsize=8)
axes[1].set_ylabel("u")
axes[1].set_xlabel("t (s)")
axes[0].set_title("Deadbeat for ZOH + 1/(s(s+1)), T = 1 s")
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch13_deadbeat.png"), dpi=150)
display(fig)
plt.close(fig)

In [ ]:
# ---------------------------------------------------------------------------
# 6. Digital PID, positional form, on G_p = 4/((s+1)(s+4))
# ---------------------------------------------------------------------------
print("\n=== Digital PID ===")

In [ ]:
Kp, Ki, Kd, T, tau_f = 4.0, 4.0, 0.75, 0.05, 0.05
alpha, gD = tau_f / (tau_f + T), Kd / (tau_f + T)
A_p = np.array([[-1.0, 1.0], [0.0, -4.0]])  # x1 = c, x2 = output of 4/(s+4)
B_p = np.array([0.0, 4.0])
C_p = np.array([1.0, 0.0])
Phi, Gam = zoh_matrices(A_p, B_p, T)
x = np.zeros(2)
uI = uD = e_old = 0.0
ck, uk = [], []

In [ ]:
for k in range(int(4 / T) + 1):
    c_now = C_p @ x
    e = 1.0 - c_now
    uI = uI + Ki * T * e                       # integral, backward Euler
    uD = alpha * uD + gD * (e - e_old)         # filtered derivative, backward Euler
    u = Kp * e + uI + uD                       # positional form
    e_old = e
    ck.append(c_now)
    uk.append(u)
    x = Phi @ x + Gam * u

In [ ]:
ck = np.array(ck)
print(f"u(0) = {uk[0]:.2f}; peak {ck.max():.4f} at t = {ck.argmax() * T:.2f} s")
t = np.linspace(0, 4, 400)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(t, 1 - np.exp(-2 * t) + t * np.exp(-2 * t), "k--", label="analog ideal PID")
ax.plot(np.arange(len(ck)) * T, ck, "r.", ms=4, label="digital PID samples")
ax.axhline(1, color="gray", lw=0.5)
ax.set_xlabel("t (s)")
ax.set_ylabel("c")
ax.legend()
ax.set_title("PID: Kp = 4, Ki = 4, Kd = 0.75, filter 0.05 s, T = 0.05 s")
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "ch13_pid.png"), dpi=150)
display(fig)
plt.close(fig)

print(f"\nplots saved in {FIGDIR}")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**One analog lead, two digital versions.** The lead $G_c(s) = 4(s+1)/(s+2)$ around
$G_p(s) = 1/(s(s+1))$ gives the analog loop $4/(s^2 + 2s + 4)$: poles $-1 \pm j\sqrt3$,
16.3% overshoot. Here the computer runs it as $D(z)$, behind a zero-order hold. Tustin's rule is
always shown (orange); pick a second method to compare (blue). At $T = 0.2$ s every method is
close (Tustin: 25.7% overshoot of the samples). Push $T$ toward 1 s and watch the hold's lag eat
the phase margin: the samples of the Tustin loop overshoot by 82%.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
METHODS = ["ZOH equivalent", "matched pole-zero", "backward Euler", "forward Euler"]
T_SHOW, T_LONG = 14.0, 40.0


def _controller(method, T):
    """D(z) = (d1 z + d0)/(z - e) for Gc(s) = 4(s+1)/(s+2)."""
    if method == "Tustin":                             # s = (2/T)(z-1)/(z+1)
        c = 2.0 / T
        num, den = 4 * np.array([c + 1, 1 - c]), np.array([c + 2, 2 - c])
    elif method == "ZOH equivalent":                   # 4 - 2/(s+2), step invariant
        q = np.exp(-2 * T)
        num, den = np.array([4.0, -2 - 2 * q]), np.array([1.0, -q])
    elif method == "matched pole-zero":                # z = e^{sT}, DC gain 2
        num, den = 2 * (1 + np.exp(-T)) * np.array([1.0, -np.exp(-T)]), np.array([1.0, -np.exp(-2 * T)])
    elif method == "backward Euler":                   # s = (z-1)/(Tz)
        num, den = 4 * np.array([1 + T, -1.0]), np.array([1 + 2 * T, -1.0])
    else:                                              # forward Euler, s = (z-1)/T
        num, den = 4 * np.array([1.0, T - 1]), np.array([1.0, 2 * T - 1])
    return num / den[0], den / den[0]


def _simulate(num, den, T):
    """Unit step through the loop; the plant 1/(s(s+1)) is exact between the samples."""
    p = np.exp(-T)
    b1 = T - 1 + p
    n = int(np.ceil(T_LONG / T))
    tau = np.linspace(0, T, 11)[:-1]
    x1 = x2 = 0.0
    u_old = e_old = 0.0
    ck, tf, cf = np.zeros(n + 1), [], []
    for k in range(n + 1):
        ck[k] = x1
        e = 1.0 - x1
        u = -den[1] * u_old + num[0] * e + num[1] * e_old
        u_old, e_old = u, e
        if k * T <= T_SHOW:
            tf.append(k * T + tau)
            cf.append(x1 + x2 * (1 - np.exp(-tau)) + u * (tau - 1 + np.exp(-tau)))
        x1, x2 = x1 + x2 * (1 - p) + u * b1, x2 * p + u * (1 - p)
    return ck, np.concatenate(tf), np.concatenate(cf)


def _closed_loop_poles(num, den, T):
    p = np.exp(-T)
    gn, gd = np.array([T - 1 + p, 1 - p - T * p]), np.array([1.0, -(1 + p), p])
    return np.roots(np.polyadd(np.polymul(den, gd), np.polymul(num, gn)))


def explore(T=0.2, compare="ZOH equivalent"):
    t = np.linspace(0, T_SHOW, 600)
    c_analog = 1 - np.exp(-t) * (np.cos(np.sqrt(3) * t) + np.sin(np.sqrt(3) * t) / np.sqrt(3))
    target = np.exp(T * np.array([-1 + 1j * np.sqrt(3), -1 - 1j * np.sqrt(3)]))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [2.1, 1]})
    ax1.plot(t, c_analog, "--", color=GRAY, lw=1.5, label="analog loop")
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(target.real, target.imag, "+", color=GRAY, ms=14, mew=2, label="$e^{sT}$ of analog poles")
    lines = []
    worst = 0.0
    for method, col, mk in [(compare, BLUE, "s"), ("Tustin", ORANGE, "o")]:
        num, den = _controller(method, T)
        ck, tf, cf = _simulate(num, den, T)
        poles = _closed_loop_poles(num, den, T)
        rmax = np.max(np.abs(poles))
        worst = max(worst, rmax)
        ks = np.arange(len(ck))
        keep = ks * T <= T_SHOW
        ax1.plot(tf, cf, color=col, lw=1.4)
        ax1.plot(ks[keep] * T, ck[keep], mk, color=col, ms=5 if T >= 0.3 else 3, label=method)
        ax2.plot(poles.real, poles.imag, "x", color=col, ms=10, mew=2.2, label=method)
        os_txt = f"{100 * (ck.max() - 1):.1f}%" if rmax < 1 else "unstable"
        lines.append(f"{method:18s} D(z) = {num[0]:.4f}(z {num[1] / num[0]:+.4f})/(z {den[1]:+.4f});"
                     f"  max |pole| = {rmax:.3f};  overshoot of the samples {os_txt}")
    ax1.axhline(1, color="k", lw=0.6)
    ax1.set_xlim(0, T_SHOW)
    ax1.set_ylim(-0.1, 2.2)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("c")
    ax1.set_title(f"unit step with the lead run at T = {T:g} s", fontsize=11)
    ax1.legend(fontsize=10, loc="lower right")
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    lim = max(1.2, 1.05 * worst)
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("closed-loop poles", fontsize=11)
    ax2.legend(fontsize=10, loc="lower left")
    fig.tight_layout()

    print(f"analog: 16.3% overshoot;  its poles -1 +- j1.732 map to {target[0].real:.4f} +- j{target[0].imag:.4f}")
    for line in reversed(lines):
        print(line)
    plt.show()


try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(explore,
             T=FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, description="T (s)",
                           continuous_update=False),
             compare=Dropdown(options=METHODS, value="ZOH equivalent", description="compare"))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 13.** These cells go with the Hands-on exercises at the end of
Chapter 13. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
# ---------------------------------------------------------------- H1
def h1_deadbeat_motor(T=0.1):
    """H1: Gp = a/(s+a), a = 10; deadbeat F = 1/z; u(0) = 1/(1-p), then u = 1; c(t) between samples."""
    a = 10.0
    p = np.exp(-a * T)
    n = int(np.ceil(0.5 / T)) + 1
    u = np.r_[1 / (1 - p), np.ones(n - 1)]
    t, c, ck = [], [], [0.0]
    x = 0.0
    for k in range(n):
        tau = np.linspace(0, T, 40)
        t.append(k * T + tau)
        c.append(u[k] + (x - u[k]) * np.exp(-a * tau))   # exact solution with u held
        x = u[k] + (x - u[k]) * p
        ck.append(x)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
    ax1.plot(np.concatenate(t), np.concatenate(c), color=BLUE, lw=1.4, label="speed c(t)")
    ax1.plot(T * np.arange(n + 1), ck, "o", color=ORANGE, ms=4, label="samples")
    ax1.axhline(1, color=GRAY, lw=0.6)
    ax1.set_ylim(0, 1.3)
    ax1.legend(loc="lower right", fontsize=9)
    ax1.set_title(f"T = {1000 * T:.1f} ms: u(0) = {u[0]:.2f}, settled at t = T", fontsize=11)
    ax2.step(T * np.arange(n + 1), np.r_[u, u[-1]], where="post", color=ORANGE, lw=1.4)
    ax2.axhline(2, color=TEAL, ls="--", lw=1, label="drive limit |u| = 2")
    ax2.set_ylim(0, max(3, 1.1 * u[0]))
    ax2.set_xlabel("t (s)")
    ax2.set_ylabel("u")
    ax2.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_hold_lag(T=0.05):
    """H2: Gc = 566(s+5)/(s+20), Tustin prewarped at 20 rad/s, on 1/(s(s+5)) with a ZOH: exact sampled PM."""
    K, a, wc = 566.0, 5.0, 20.0
    p = np.exp(-a * T)
    b1, b0 = (a * T - 1 + p) / a ** 2, (1 - p - a * T * p) / a ** 2   # ZOH of 1/(s(s+a))
    c1 = wc / np.tan(wc * T / 2)
    w = np.linspace(0.5, np.pi / T * 0.999, 4000)
    z = np.exp(1j * w * T)
    D = K * ((c1 + 5) * z - (c1 - 5)) / ((c1 + 20) * z - (c1 - 20))
    G = (b1 * z + b0) / ((z - 1) * (z - p))
    L = D * G
    La = K / (1j * w * (1j * w + 20))
    i = np.argmin(abs(abs(L) - 1))
    ia = np.argmin(abs(abs(La) - 1))
    pm = 180 + np.degrees(np.angle(L[i]))
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.semilogx(w, np.degrees(np.unwrap(np.angle(La))), color=GRAY, lw=1.4, label="analog loop")
    ax.semilogx(w, np.degrees(np.unwrap(np.angle(L))), color=ORANGE, lw=1.4, label="sampled loop")
    ax.axhline(-180, color=GRAY, lw=0.6)
    ax.axvline(w[i], color=ORANGE, ls=":", lw=1)
    ax.set_ylim(-270, -80)
    ax.set_xlabel("frequency (rad/s)")
    ax.set_ylabel("phase (deg)")
    ax.set_title(f"T = {1000 * T:.1f} ms: PM analog {180 + np.degrees(np.angle(La[ia])):.1f}, sampled {pm:.1f} deg, "
                 f"estimate {45 - np.degrees(wc * T / 2):.1f} deg", fontsize=11)
    ax.legend(loc="lower left", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_windup(umax=0.25, form="positional"):
    """H3: tank level, Gp = 1/s, T = 0.1 s, PI Kp = 1, Ki = 0.25, |u| <= umax, step 0 -> 1 m."""
    T, Kp, Ki, n = 0.1, 1.0, 0.25, 400
    h = np.zeros(n + 1)
    u = np.zeros(n)
    e = np.zeros(n)
    uI = uprev = eprev = 0.0
    for k in range(n):
        e[k] = 1.0 - h[k]
        if form == "positional":
            uI += Ki * T * e[k]
            u[k] = np.clip(Kp * e[k] + uI, -umax, umax)
        else:
            u[k] = np.clip(uprev + Kp * (e[k] - eprev) + Ki * T * e[k], -umax, umax)
        uprev, eprev = u[k], e[k]
        h[k + 1] = h[k] + T * u[k]
    pos, neg = T * e[e > 0].sum(), T * e[e < 0].sum()
    t = T * np.arange(n + 1)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
    ax1.plot(t, h, color=BLUE, lw=1.4)
    ax1.axhline(1, color=GRAY, lw=0.6)
    ax1.fill_between(t, h, 1, where=h < 1, color=ORANGE, alpha=0.15)
    ax1.fill_between(t, h, 1, where=h > 1, color=TEAL, alpha=0.25)
    ax1.set_ylim(0, 1.8)
    ax1.set_ylabel("level (m)")
    ax1.set_title(f"{form}, umax = {umax:.2f} m/s: overshoot {100 * max(h.max() - 1, 0):.1f}%, "
                  f"error areas +{pos:.2f} / {neg:.2f} m s", fontsize=11)
    ax2.step(t[:-1], u, where="post", color=ORANGE, lw=1.4)
    ax2.set_ylabel("u (m/s)")
    ax2.set_xlabel("t (s)")
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_derivative_filter(tau_f=0.03):
    """H4: D term Kd/(tau_f+T) (z-1)/(z-alpha), T = 0.01 s, Kd = 0.5; gain up to half the sampling frequency."""
    T, Kd, wc = 0.01, 0.5, 4.0
    alpha = tau_f / (tau_f + T)
    w = np.linspace(0.1, np.pi / T, 3000)
    z = np.exp(1j * w * T)
    DD = Kd / (tau_f + T) * (z - 1) / (z - alpha)
    D0 = Kd / T * (z - 1) / z                         # no filter
    nyq = 2 * Kd / (2 * tau_f + T)
    lag = np.degrees(np.arctan(wc * tau_f))
    ok = nyq <= 20 and lag <= 10
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.semilogx(w, abs(D0), color=GRAY, lw=1.2, label="no filter")
    ax.semilogx(w, abs(DD), color=ORANGE, lw=1.6, label=f"tau_f = {1000 * tau_f:.0f} ms")
    ax.axhline(20, color=TEAL, ls="--", lw=1, label="noise limit 20")
    ax.set_xlabel("frequency (rad/s)")
    ax.set_ylabel("gain of the D term")
    ax.set_title(f"gain at half the sampling frequency {nyq:.1f}, filter lag at 4 rad/s {lag:.1f} deg: "
                 f"{'PASS' if ok else 'FAIL'}", fontsize=11)
    ax.legend(loc="upper left", fontsize=9)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(h1_deadbeat_motor, T=FloatSlider(value=0.1, min=0.01, max=0.2, step=0.001, readout_format=".3f",
                                              description="T (s)", continuous_update=False))
    interact(h2_hold_lag, T=FloatSlider(value=0.05, min=0.002, max=0.08, step=0.001, readout_format=".3f",
                                        description="T (s)", continuous_update=False))
    interact(h3_windup, umax=FloatSlider(value=0.25, min=0.1, max=2.0, step=0.01, description="umax (m/s)",
                                         continuous_update=False),
             form=Dropdown(options=["positional", "velocity"], value="positional", description="form"))
    interact(h4_derivative_filter, tau_f=FloatSlider(value=0.03, min=0.0, max=0.08, step=0.001,
                                                     readout_format=".3f", description="tau_f (s)",
                                                     continuous_update=False))
except ImportError:
    h1_deadbeat_motor()
    h2_hold_lag()
    h3_windup()
    h4_derivative_filter()